# Week 3 — SQL (Lakehouse) 
*Catalog/Schema fixed to `bh2026-winford-uc-dev.bh_lakehouse`*
*Each step below is isolated in its own cell to see results per query.*

-- SQL Lakehouse Design: Bronze, Silver, Gold Layers

-- This script demonstrates the creation and population of Bronze, Silver, and Gold layers
-- using standard SQL. 

-- 1. Bronze Layer: Raw Data Ingestion
-- This layer holds the raw, untransformed data directly from the source. Data types
-- are often kept as strings to preserve the original format before cleaning.

## Initialisation & Setup

In [0]:
-- Create Catalog & Schema (explicit)
-- CREATE CATALOG IF NOT EXISTS `bh2026-winford-uc-dev`;
USE CATALOG `bh2026-winford-uc-dev`;
CREATE SCHEMA IF NOT EXISTS bh_lakehouse;
USE SCHEMA bh_lakehouse;
SELECT current_catalog() AS current_catalog, current_schema() AS current_schema;

## 1. Create Bronze Layer Tables then Insert values using using SQL DDL/DMLclauses

In [0]:
-- Create Bronze Customers Table
-- CREATE TABLE bronze_customers (
--     customer_id VARCHAR(10),
--     customer_name VARCHAR(100),
--     email VARCHAR(100),
--     city VARCHAR(50)
-- );

-- Insert raw data into Bronze Customers
INSERT INTO bronze_customers (customer_id, customer_name, email, city) VALUES
('C001', 'Alice Smith', 'alice.s@example.com', 'New York'),
('C002', 'Bob Johnson', 'bob.j@example.com', 'Los Angeles'),
('C003', 'Charlie Brown', 'charlie.b@example.com', 'Chicago'),
('C004', 'Diana Prince', 'diana.p@example.com', 'Miami'),
('C005', 'Eve Adams', 'eve.a@example.com', 'Houston');

In [0]:
-- Create Bronze Product table
Create OR REPLACE TABLE bronze_products (
    product_id VARCHAR(10),
    product_name VARCHAR(100),
    product_category VARCHAR(50),
    price VARCHAR(20)
);

-- insert values

INSERT INTO bronze_products (product_id, product_name, product_category, price) VALUES
('P001', 'Laptop', 'Electronics', '1200.00'),
('P002', 'Mouse', 'Electronics', '25.00'),
('P003', 'Keyboard', 'Electronics', '50.00'),
('P004', 'Monitor', 'Electronics', '300.00'),
('P005', 'Headphones', 'Electronics', '80.00');

In [0]:
-- Create Bronze Orders Table
CREATE TABLE bronze_orders (
    order_id VARCHAR(10),
    customer_id VARCHAR(10),
    order_date VARCHAR(20), -- Raw date might be string
    status VARCHAR(50)
);

-- Insert raw data into Bronze Orders
INSERT INTO bronze_orders (order_id, customer_id, order_date, status) VALUES
('O001', 'C001', '2023-01-10', 'Completed'),
('O002', 'C002', '2023-01-11', 'Pending'),
('O003', 'C001', '2023-01-12', 'Completed'),
('O004', 'C003', '2023-01-13', 'Shipped'),
('O005', 'C004', '2023-01-14', 'Completed');

In [0]:
-- Create Bronze Order Items Table
CREATE TABLE bronze_order_items (
    order_item_id VARCHAR(10),
    order_id VARCHAR(10),
    product_id VARCHAR(10),
    quantity VARCHAR(10), -- Raw quantity might be string
    unit_price VARCHAR(20) -- Raw price might be string
);

-- Insert raw data into Bronze Order Items
INSERT INTO bronze_order_items (order_item_id, order_id, product_id, quantity, unit_price) VALUES
('OI001', 'O001', 'P001', '1', '1200.00'),
('OI002', 'O001', 'P002', '2', '25.00'),
('OI003', 'O002', 'P003', '1', '75.00'),
('OI004', 'O003', 'P004', '1', '300.00'),
('OI005', 'O004', 'P005', '3', '50.00');

In [0]:
-- Create Bronze Payments Table
CREATE TABLE bronze_payments (
    payment_id VARCHAR(10),
    order_id VARCHAR(10),
    payment_date VARCHAR(20), -- Raw date might be string
    amount VARCHAR(20), -- Raw amount might be string
    payment_method VARCHAR(50)
);

-- Insert raw data into Bronze Payments
INSERT INTO bronze_payments (payment_id, order_id, payment_date, amount, payment_method) VALUES
('PM001', 'O001', '2023-01-10', '1250.00', 'Credit Card'),
('PM002', 'O003', '2023-01-12', '300.00', 'Debit Card'),
('PM003', 'O004', '2023-01-13', '150.00', 'PayPal'),
('PM004', 'O005', '2023-01-14', '500.00', 'Credit Card'),
('PM005', 'O002', '2023-01-11', '75.00', 'Credit Card');

## Create Silver Layer `Tables`

## 2. Create all the Silver Layer Tables using SQL clauses, Functions and CTEs

## Silver Layer: Cleaned and Conformed Data

This layer applies basic transformations, data type conversions, and standardization.
\
We'll create a conformed 'silver_orders_details' table by joining relevant bronze tables,
\
casting data types, and calculating derived fields.

In [0]:
-- DROP TABLE IF EXISTS silver_products;

-- CREATE OR REPLACE TABLE silver_orders_details (
--     order_id VARCHAR(10),
--     customer_id VARCHAR(10),
--     customer_name VARCHAR(100),
--     order_date DATE,
--     product_id VARCHAR(10),
--     product_name VARCHAR(100),
--     category VARCHAR(50),
--     quantity INT,
--     unit_price DECIMAL(10, 2),
--     total_item_price DECIMAL(10, 2),
--     order_status VARCHAR(50),
--     payment_method VARCHAR(50),
--     payment_amount DECIMAL(10, 2),
--     payment_date DATE
-- );

INSERT INTO silver_orders_details (
    order_id, customer_id, customer_name, order_date,
    product_id, product_name, category, quantity, unit_price, total_item_price,
    order_status, payment_method, payment_amount, payment_date
)
SELECT
    o.order_id,
    c.customer_id,
    c.customer_name,
    CAST(o.order_date AS DATE) AS order_date,
    p.product_id,
    p.product_name,
    p.product_category,
    CAST(oi.quantity AS INT) AS quantity,
    CAST(oi.unit_price AS DECIMAL(10, 2)) AS unit_price,
    CAST(oi.quantity AS INT) * CAST(oi.unit_price AS DECIMAL(10, 2)) AS total_item_price,
    o.status AS order_status,
    pm.payment_method,
    CAST(pm.amount AS DECIMAL(10, 2)) AS payment_amount,
    CAST(pm.payment_date AS DATE) AS payment_date
FROM
    bronze_orders o
JOIN
    bronze_customers c ON o.customer_id = c.customer_id
JOIN
    bronze_order_items oi ON o.order_id = oi.order_id
JOIN
    bronze_products p ON oi.product_id = p.product_id
LEFT JOIN -- Use LEFT JOIN for payments as an order might not have a payment recorded yet
    bronze_payments pm ON o.order_id = pm.order_id;

## 3. Create Gold Layer `Table/View`

Gold Layer: Curated and Aggregated Data.
\
This layer provides highly refined, aggregated data for business intelligence and reporting, this creates a summary table for monthly sales by category.

In [0]:
CREATE TABLE gold_monthly_sales_by_category (
    sales_month DATE,
    category VARCHAR(50),
    total_sales DECIMAL(18, 2),
    total_orders INT,
    unique_customers INT
);

INSERT INTO gold_monthly_sales_by_category (
    sales_month, category, total_sales, total_orders, unique_customers
)
SELECT
    DATE_TRUNC('month', order_date) AS sales_month, -- Use appropriate date truncation for your SQL dialect (e.g., TRUNC(order_date, 'MM') for Oracle, DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1) for SQL Server)
    category,
    SUM(total_item_price) AS total_sales,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM
    silver_orders_details
GROUP BY
    DATE_TRUNC('month', order_date),
    category
ORDER BY
    sales_month, category;

-- Example Query on Gold Layer
SELECT * FROM gold_monthly_sales_by_category;

-- Example Query on Silver Layer
SELECT
    customer_name,
    order_id,
    order_date,
    SUM(total_item_price) AS order_total
FROM
    silver_orders_details
GROUP BY
    customer_name, order_id, order_date
ORDER BY
    order_date DESC;


### Wk3 Lakehouse Q&As
 
1. Basic SELECT
Retrieve all columns from the bronze_customers table.
2. Selecting Specific Columns
Display product_name, category, and price from the bronze_products table.
3. ORDER BY
Show all products ordered by price from highest to lowest.
4. WHERE Filtering
Retrieve all orders where the order status is 'Completed'.
5. AND Condition
Find all products where category = 'Electronics' AND price > 100.
6. LIKE Operator
Find all customers whose names start with the letter 'A'.
7. Aggregation
Calculate the total payment amount from the bronze_payments table.
8. GROUP BY
Count how many orders each customer has placed using the bronze_orders table.
9. INNER JOIN
Join bronze_orders with bronze_customers to display order_id, customer_name, order_date, and status.
10. Multi-Table JOIN Challenge
Using bronze_orders, bronze_order_items, and bronze_products, create a query that displays order_id, product_name, quantity, unit_price, and total_item_price where total_item_price = quantity * unit_price.


In [0]:
-- Q1 Basic SELECT Retrieve all columns from the bronze_customers table.

SELECT * FROM bronze_customers
LIMIT 10;

In [0]:
-- Q2 Selecting Specific Columns Display product_name, category, and price from the bronze_products table.

-- SELECT * FROM bronze_products
-- LIMIT 10;

SELECT product_name, product_category, price 
FROM bronze_products
LIMIT 10;

In [0]:
-- Q3. ORDER BY Show all products ordered by price from highest to lowest.

SELECT * FROM bronze_products
ORDER BY price DESC
LIMIT 10;


In [0]:
-- Q.4 WHERE Filtering Retrieve all orders where the order status is 'Completed'.
SELECT * 
FROM bronze_orders
WHERE status = 'Completed';

In [0]:
-- Q5. AND Condition: Find all products where category = 'Electronics' AND price > 100.

SELECT * 
FROM bronze_products
WHERE product_category = 'Electronics' 
  AND CAST(price AS DECIMAL(10,2)) > 100;

How this logic is useful in an analytics meeting
The WHERE ... AND pattern is a segmentation filter — for example this query answers "Which high-value products do we carry in a specific category?"

I would this information to present in a meeting for discussion:

1. Pricing strategy — Quickly identify premium-tier items to discuss markdown opportunities, bundling deals, or margin analysis.
2. Inventory prioritisation — High-price items in a key category (Electronics) typically carry more stock risk; this helps focus reorder/forecasting discussions.
3. Marketing targeting — If planning a campaign for premium electronics, this query instantly gives you the product shortlist.
4. Revenue concentration — Showing stakeholders that only **2 of 5** electronics exceed £100 highlights how revenue may be concentrated in a few Stock Items/SKUs.
\
#### although the query is a basic query, it shows by combining conditions with AND lets me slice data into actionable segments rather than reviewing everything — which is exactly what decision-makers need in a time-limited meeting!

In [0]:
-- Q6. LIKE Operator: Find all customers whose names start with the letter 'A'.

SELECT * 
FROM bronze_customers
WHERE customer_name LIKE 'A%';

The LIKE 'A%' pattern "Which customers match a specific naming pattern?"

1. Customer segmentation by demographics (e.g., names, countries etc starting with certain letters may correlate with geographic markets).
2. Data quality auditing — Quickly spot naming inconsistencies, duplicates, or formatting issues (e.g., "Alice" vs "alice" vs "ALICE") before they corrupt downstream reports.
3. Quick lookup during live meetings — When a stakeholder says "Pull up that customer whose name starts with A…", ...

#### The LIKE operator is useful when I dont have the exact values. So when combined with % (wildcard), it turns a vague request into a precise result set, making it a go-to tool for ad-hoc exploration during meetings.

In [0]:
-- Q7. Aggregation: Calculate the total payment amount from the bronze_payments table.

SELECT 
  SUM(CAST(amount AS DECIMAL(10,2))) AS total_payment_amount
FROM bronze_payments;

The SUM() aggregate allows me to determine "How much total revenue have we collected?"

1. Revenue snapshot 
2. Cash flow visibility 
3. Reconciliation — Compare total_payment_amount (£2,275) against total order value to quickly spot collection gaps — are there unpaid orders or partial payments?
4. Trend baseline — This single number becomes the starting point for time-based breakdowns (weekly, monthly) so you can show growth or decline in future meetings.

#### aggregation condenses thousands of rows into a single actionable number — exactly what decision-makers need to assess overall business health without wading through individual transactions.

In [0]:
-- Q8. GROUP BY and COUNT Count the number of products in each category.

SELECT 
  product_name, 
  COUNT(*) AS total_orders
FROM bronze_products
GROUP BY product_name
ORDER BY total_orders DESC;

GROUP BY summarises categories/columns — "How is activity distributed across groups?"

1. Customer engagement ranking — Instantly see which customers order most frequently, this could help prioritise loyalty programmes or account management attention.
2. Workload distribution — If C001 has 2 orders while others have 1, it reveals repeat-buyer behaviour vs. one-time purchasers — critical for churn and retention discussions eg. as discussed in DE week 4 zoom 
3. Resource allocation — High-order customers may need dedicated support; this query identifies them without manual counting.
4. Trend detection When combined with time periods (e.g., GROUP BY customer_id, month), it reveals whether ordering patterns are growing or declining per customer as discussed in DE week 4 zoom.

#### GROUP BY + COUNT(*) turns thousands of individual transactions into a ranked summary — the exact format stakeholders need to make decisions about who matters most.

In [0]:
-- Q9. INNER JOIN Join bronze_orders with bronze_customers to display order_id, customer_name, order_date, and status.

SELECT o.order_id, c.customer_name, o.order_date, o.status 
FROM bronze_orders o
INNER JOIN bronze_customers c 
ON o.customer_id = c.customer_id;

How INNER JOIN is useful in an analytics meeting
The INNER JOIN answers: "What does this transaction data look like when we enrich it with human-readable context?"

Raw order data contains IDs (C001, C002) — meaningless to stakeholders. The JOIN replaces those IDs with customer names, making the data presentation-ready.

Practical use cases you could present:

Order accountability — Stakeholders can instantly see who placed each order rather than decoding IDs, making discussions actionable ("Alice has two completed orders — let's reward her loyalty").
Status review meetings — Combining customer_name + status lets you discuss follow-ups by name: "Bob's order is still Pending — who owns resolution?"
Relationship mapping — The JOIN proves which customers have matching records in both tables. If a customer had no orders, they'd be excluded — that absence itself is a useful insight (dormant customers).
Foundation for richer analysis — This two-table JOIN is the building block for the multi-table JOIN in Q10 (your Silver layer pattern). In meetings, you'd layer on products and payments to give the full picture.

#### INNER JOIN is the bridge between normalised database design (efficient storage) and denormalised business reporting (human-readable insights). It's how raw IDs become actionable intelligence in a meeting room.


### INNER JOIN vs SELF JOIN

Aspect          | INNER JOIN                         | SELF JOIN                          |
\
Tables involved | Two **different** tables           | The **same** table joined to itself |
\
Purpose         | Combine related data across tables | Compare rows within the same table |
\
Syntax          | `FROM table_a JOIN table_b ON ...` | `FROM table_a t1 JOIN table_a t2 ON ...` |
\
Aliases         | Optional (but recommended)         | **Required** (to distinguish the two copies) |

`bronze_orders` to `bronze_customers` to combine order IDs with customer names. Two separate entities, one shared key (`customer_id`).

Wherreas, if I use 'self join' this would be used as discussed for DE Wk4 zoom *"How do rows in the same table relate to each other?"*

using my `bronze_orders` table — find customers who placed more than one order:

```sql
SELECT o1.order_id AS first_order, o2.order_id AS second_order, o1.customer_id
FROM bronze_orders o1
INNER JOIN bronze_orders o2 
  ON o1.customer_id = o2.customer_id
  AND o1.order_id < o2.order_id;
```

= Alice (C001) since she has orders O001 and O003.

* **INNER JOIN** = "Show me this data combined/enriched with context from another source"
* **SELF JOIN** = "Show me patterns *within* the same dataset" (repeat buyers, manager-employee hierarchies, sequential events)

##### so if stakeholders ask me questions like *"Which customers ordered more than once or would be classed as repeat customers?"* or *"Which orders happened on consecutive days?"* — these type of questions that compare a table against itself!

In [0]:
-- Q10. Multi-Table JOIN: Display order_id, product_name, quantity, unit_price, and total_item_price
-- using bronze_orders, bronze_order_items, and bronze_products.

SELECT 
  o.order_id,
  p.product_name,
  CAST(oi.quantity AS INT) AS quantity,
  CAST(oi.unit_price AS DECIMAL(10,2)) AS unit_price,
  CAST(oi.quantity AS INT) * CAST(oi.unit_price AS DECIMAL(10,2)) AS total_item_price
FROM bronze_orders o
INNER JOIN bronze_order_items oi ON o.order_id = oi.order_id
INNER JOIN bronze_products p ON oi.product_id = p.product_id
ORDER BY o.order_id;

I created this Multi-Table JOIN which can be used for answering questions by stakeholders such as "What exactly did each order contain, or what was each line item worth?"

There three tables chained/joined together: bronze_orders → bronze_order_items → bronze_products, combining transactional data with product context and calculating a derived metric (total_item_price) on the fly.

1. Revenue decomposition — Stakeholders can see that O001 generated £1,250 total (£1,200 Laptop + £50 Mouse), revealing which products drive order value. it can be seen that One Laptop contributes more than all the entire orders from O002–O004 combined!
2. Product performance — The calculated total_item_price shows that Headphones (3 × £50 = £150) outperform Keyboard (1 × £75 = £75) by volume despite a lower unit price — critical for inventory and bundling decisions.
3. Order composition analysis — O001 has multiple items while others have one. This identifies cross-sell opportunities ("customers buying Laptops also buy Mice").
4. Data validation — By computing quantity * unit_price live, can be used to identify any mismatch data quality issues worth investigating before they reach finance reports/board meeting.

#### My multi-table Join table/View is very similar to my Silver Layer table, which combined and normalised my Bronze tables to become a single 'denormalised' Gold table/view that business stakeholders Downstream in the pipeline such as Data Analyst / Data Scientist or other Users can read without understanding database relationships! which is the purpose as a Data Engineer to extract and provide exact transformation from "engineer-friendly storage" to "business-friendly reporting."